# Read-strategy tests for `time=1` chunking (PACE sandbox)

PACE OCI is stored one day per chunk (`time=1`). Reading long time spans from that layout is slow, so
before touching the real PACE store we develop and measure a read strategy on `IO_rechunked.zarr`,
which has the same `time=1` spatial chunking in a small, public, no-auth dataset.

Each section isolates **one lever** and times it against a baseline. We time raw block loads (pure I/O),
so this notebook needs only `xarray` + `dask`, not the model or the standardization step. Numbers will
vary run to run and with filesystem caching; re-run a cell a couple of times and watch the ratios, not
the absolute seconds.

In [ ]:
import os
from time import perf_counter
import numpy as np
import xarray as xr
import dask

## Open both layouts

`IO.zarr` is the original (`time=100`, whole spatial domain). `IO_rechunked.zarr` is the PACE-like
layout (`time=1`, `lat=40`, `lon=56`). Same data, same Arabian Sea crop. The printout shows how the
identical region is split into chunks very differently on disk.

In [ ]:
lat_min, lat_max = 5, 31
lon_min, lon_max = 42, 80

def open_crop(path):
    ds = xr.open_zarr(path, chunks={})
    return ds.sel(lat=slice(lat_max, lat_min), lon=slice(lon_min, lon_max))

orig = open_crop(os.path.expanduser("~/shared/mind_the_chl_gap/IO.zarr"))
rech = open_crop("/home/jovyan/shared-public/mindthegap/data/IO_rechunked.zarr")

for name, ds in [("original", orig), ("rechunked", rech)]:
    cs = ds["CHL_cmes-level3"].chunksizes
    print(f"{name:10s} time={tuple(cs['time'])[:3]}...  lat={tuple(cs['lat'])}  lon={tuple(cs['lon'])}")

## Timing helper

`time_load` returns wall-clock seconds to pull a block into memory. It reads only the raw variables
(the actual on-disk reads), under a chosen dask scheduler so we can compare serial vs parallel.

In [ ]:
RAW = ["CHL_cmes-level3", "CHL_cmes-cloud", "u_wind", "v_wind", "sst", "air_temp"]

def time_load(ds, tsel, lat=slice(None), lon=slice(None), scheduler="threads"):
    sub = ds.isel(time=tsel, lat=lat, lon=lon)[RAW]
    with dask.config.set(scheduler=scheduler):
        t = perf_counter()
        sub.load()
        return perf_counter() - t

## Baseline: the same 100-day block, two layouts

A 100-day whole-domain block is **one** on-disk chunk in the original file, and ~900 tiny chunks in
the rechunked file (100 days x ~9 spatial tiles). This is the gap we are trying to close.

In [ ]:
t_orig = time_load(orig, slice(0, 100))
t_rech = time_load(rech, slice(0, 100))
print(f"original  100-day block: {t_orig:6.2f} s")
print(f"rechunked 100-day block: {t_rech:6.2f} s   ({t_rech / t_orig:.1f}x slower)")

## Lever 1: parallelize the reads

The rechunked block is hundreds of small reads. `synchronous` does them one at a time; `threads`
issues them concurrently so the per-read latency overlaps. This is the biggest expected lever for
`time=1` data.

In [ ]:
t_serial = time_load(rech, slice(0, 100), scheduler="synchronous")
t_thread = time_load(rech, slice(0, 100), scheduler="threads")
print(f"rechunked serial  : {t_serial:6.2f} s")
print(f"rechunked threaded: {t_thread:6.2f} s   ({t_serial / t_thread:.1f}x faster)")

## Lever 2: avoid the time-gather

Loading 100 days at once makes dask assemble 100 one-day chunks into a single block. Loading a few
days at a time keeps reads aligned to how the data is stored: same total reads, but no big re-assembly
and far less memory held at once.

In [ ]:
t_100  = time_load(rech, slice(0, 100))
t_10x10 = sum(time_load(rech, slice(k * 10, k * 10 + 10)) for k in range(10))
print(f"one 100-day load : {t_100:6.2f} s")
print(f"ten 10-day loads : {t_10x10:6.2f} s")

## Lever 3: cache a spatial tile in RAM (the pattern that scales)

The scaling-friendly version of "just load everything": hold one spatial tile's time series in memory,
train on all its days, then move to the next tile. Reads are paid once per tile, not once per epoch.
Globally you size the tile so `tile x days` fits in RAM.

(A 40x56 tile is one on-disk spatial chunk on the full grid; the Arab Sea subset offsets it slightly,
so it is roughly, not exactly, one chunk. Persisting the *whole* region also works here because it is
tiny, but that does not scale, which is why we test a single tile.)

In [ ]:
tile_lat, tile_lon = slice(0, 40), slice(0, 56)

t_first = time_load(rech, slice(0, 365), lat=tile_lat, lon=tile_lon)   # a year of one tile
print(f"load 1 tile x 365 days (first time): {t_first:6.2f} s")

cached = rech.isel(time=slice(0, 365), lat=tile_lat, lon=tile_lon)[RAW].compute()  # now in RAM
t = perf_counter()
_ = cached.isel(time=200)          # any day, straight from memory
print(f"per-day access from cached tile    : {perf_counter() - t:.5f} s")

## Summary

- **Baseline gap:** rechunked `time=1` is Nx slower for a 100-day block (see above).
- **Lever 1 (parallel reads):** expected to recover most of the gap.
- **Lever 2 (avoid the time-gather):** helps memory and skips the re-assembly.
- **Lever 3 (per-tile caching):** amortizes reads across many training steps and is the pattern that
  carries to global PACE.

Whatever combination lands closest to the original-file baseline is the read strategy to bring to the
real PACE store.